In [1]:
import pandas as pd 
import numpy as np
import nltk
from nltk.corpus import stopwords
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score , confusion_matrix , classification_report
from nltk.stem import WordNetLemmatizer
stop_words = set(stopwords.words('english'))
import mlflow
import dagshub

c:\Users\nice\anaconda3\envs\temp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df=pd.read_csv("IMDB.csv")

In [3]:
def lemmatiztion(text):
    lemmatizer=WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)
def lower_case(text):
    return text.lower()
def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)
def remove_stopwords(text):
    text= text.split()
    text= [word for word in text if word not in stop_words]
    return " ".join(text)
def remove_numbers(text):
    text=text.split()
    text=[word for word in text if not word.isdigit()]
    return " ".join(text)
def remove_url(text):
    return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
def preprocess_text(text):
    text= lower_case(text)
    text= remove_punctuation(text)
    text= remove_stopwords(text)
    text= lemmatiztion(text)
    text= remove_numbers(text)
    text= remove_url(text)
    return text
df["clean_review"] = df["review"].apply(preprocess_text)
df.head()

,review,sentiment,clean_review
0,Film version of Sandra Bernhard's one-woman of...,negative,film version sandra bernhards onewoman offbroa...
1,I switched this on (from cable) on a whim and ...,positive,switched cable whim treated quite surprisealth...
2,The `plot' of this film contains a few holes y...,negative,plot film contains hole could drive massive tr...
3,"Some amusing humor, some that falls flat, some...",negative,amusing humor fall flat decent acting quite at...
4,What can you say about this movie? It was not ...,negative,say movie terrible good two day earlier watche...


In [4]:
vectorizer=TfidfVectorizer(max_features=500)
X=vectorizer.fit_transform(df["clean_review"])
y=df["sentiment"].map({"positive":1,"negative":0})
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=42)

In [5]:
mlflow.set_tracking_uri("https://dagshub.com/abdelwahab798/full-mlops-project-for-sentiment-analysis.mlflow")
dagshub.init(repo_name="full-mlops-project-for-sentiment-analysis", repo_owner="abdelwahab798",mlflow=True)
mlflow.set_experiment("TF-IDF")

Accessing as abdelwahab798

Initialized MLflow to track repo "abdelwahab798/full-mlops-project-for-sentiment-analysis"

Repository abdelwahab798/full-mlops-project-for-sentiment-analysis initialized!

2026/04/30 18:51:33 INFO mlflow.tracking.fluent: Experiment with name 'TF-IDF' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/f4958df4a52d4ec28d062112b925344c', creation_time=1777564294222, experiment_id='3', last_update_time=1777564294222, lifecycle_stage='active', name='TF-IDF', tags={}, trace_location=None, workspace='default'>

In [6]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost 
import logging

logging.basicConfig(level=logging.INFO)
models ={
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "XGBoost": XGBClassifier(),
    "Random Forest": RandomForestClassifier()
}

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred)
        }
        
        mlflow.log_metrics(metrics)
        mlflow.set_tag("model_type", name)
        logging.info(f"Accuracy: {metrics['accuracy']}")
        logging.info(f"Precision: {metrics['precision']}")
        logging.info(f"Recall: {metrics['recall']}")
        logging.info(f"F1 Score: {metrics['f1_score']}")

        if name == "XGBoost":
            mlflow.xgboost.log_model(model, artifact_path="model")
        else:
            mlflow.sklearn.log_model(model, artifact_path="model")

INFO:root:Accuracy: 0.765
INFO:root:Precision: 0.7578947368421053
INFO:root:Recall: 0.75
INFO:root:F1 Score: 0.7539267015706806
2026/04/30 18:51:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/30 18:51:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic Regression at: https://dagshub.com/abdelwahab798/full-mlops-project-for-sentiment-analysis.mlflow/#/experiments/3/runs/f2f97ca56974484d8617a51a5f9eb089
🧪 View experiment at: https://dagshub.com/abdelwahab798/full-mlops-project-for-sentiment-analysis.mlflow/#/experiments/3


INFO:root:Accuracy: 0.75
INFO:root:Precision: 0.73
INFO:root:Recall: 0.7604166666666666
INFO:root:F1 Score: 0.7448979591836735
2026/04/30 18:51:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: https://dagshub.com/abdelwahab798/full-mlops-project-for-sentiment-analysis.mlflow/#/experiments/3/runs/e240e292f88741a08f302743da8560c1
🧪 View experiment at: https://dagshub.com/abdelwahab798/full-mlops-project-for-sentiment-analysis.mlflow/#/experiments/3


INFO:root:Accuracy: 0.735
INFO:root:Precision: 0.7415730337078652
INFO:root:Recall: 0.6875
INFO:root:F1 Score: 0.7135135135135136
2026/04/30 18:52:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/30 18:52:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest at: https://dagshub.com/abdelwahab798/full-mlops-project-for-sentiment-analysis.mlflow/#/experiments/3/runs/e177008f4c344c9098653e7bae3f6801
🧪 View experiment at: https://dagshub.com/abdelwahab798/full-mlops-project-for-sentiment-analysis.mlflow/#/experiments/3
